In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Colab-ready Turkish QA: Fine-tuning vs Retrieval-Augmented QA
# Uses extractive QA with a Turkish BERT reader.
# DATA_PATH is fixed to the Google Drive location requested by the user.

import sys
import subprocess
import importlib.util

def ensure_package(pkg_name, install_name=None):
    if importlib.util.find_spec(pkg_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", install_name or pkg_name])

# Core deps
ensure_package("datasets")
ensure_package("transformers")
ensure_package("sentence_transformers", "sentence-transformers")
ensure_package("rank_bm25")
ensure_package("sklearn", "scikit-learn")
ensure_package("pandas")
ensure_package("numpy")
ensure_package("torch")

import os
import re
import json
import random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    DefaultDataCollator,
    pipeline,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

# User-requested Drive path
DATA_PATH = r"/content/drive/MyDrive/Murat_Calisma/lokalize_hibrit/combined_data.json"

MODEL_NAME = "dbmdz/bert-base-turkish-cased"
DENSE_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
MAX_LENGTH = 384
DOC_STRIDE = 128
TOP_K = 5
OUTPUT_DIR = "./turkish_qa_reader"

def normalize_text(s):
    s = str(s).strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s

def compute_em_f1(prediction, ground_truth):
    pred = normalize_text(prediction)
    gold = normalize_text(ground_truth)
    em = float(pred == gold)

    pred_tokens = pred.split()
    gold_tokens = gold.split()

    if len(pred_tokens) == 0 and len(gold_tokens) == 0:
        return em, 1.0
    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return em, 0.0

    common = {}
    for tok in pred_tokens:
        if tok in gold_tokens:
            common[tok] = min(pred_tokens.count(tok), gold_tokens.count(tok))
    num_same = sum(common.values())

    if num_same == 0:
        return em, 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)
    f1 = 2 * precision * recall / (precision + recall)
    return em, f1

def load_squad_like_json(path):
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    examples = []
    ex_id = 0
    for article in raw["data"]:
        for para in article["paragraphs"]:
            context = para["context"]
            for qa in para["qas"]:
                answers = qa.get("answers", [])
                if not answers:
                    continue
                answer_text = answers[0]["text"]
                answer_start = answers[0]["answer_start"]
                question = qa["question"]
                examples.append({
                    "id": str(ex_id),
                    "title": article.get("title", ""),
                    "context": context,
                    "question": question,
                    "answers": {
                        "text": [answer_text],
                        "answer_start": [answer_start],
                    }
                })
                ex_id += 1
    return examples

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"DATA_PATH not found: {DATA_PATH}\n"
        "Make sure Google Drive is mounted in Colab and the file exists at that exact path."
    )

examples = load_squad_like_json(DATA_PATH)
print("Total QA examples:", len(examples))

train_examples, temp_examples = train_test_split(
    examples, test_size=0.20, random_state=SEED, shuffle=True
)
val_examples, test_examples = train_test_split(
    temp_examples, test_size=0.50, random_state=SEED, shuffle=True
)
print("Train/Val/Test:", len(train_examples), len(val_examples), len(test_examples))

train_ds = Dataset.from_list(train_examples)
val_ds = Dataset.from_list(val_examples)
test_ds = Dataset.from_list(test_examples)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)

def prepare_train_features(examples):
    tokenized_examples = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_mapping = tokenized_examples.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized_examples.pop("offset_mapping")

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized_examples["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)
        sequence_ids = tokenized_examples.sequence_ids(i)
        sample_idx = sample_mapping[i]
        answers = examples["answers"][sample_idx]

        if len(answers["answer_start"]) == 0:
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        start_char = answers["answer_start"][0]
        end_char = start_char + len(answers["text"][0])

        token_start_index = 0
        while sequence_ids[token_start_index] != 1:
            token_start_index += 1

        token_end_index = len(input_ids) - 1
        while sequence_ids[token_end_index] != 1:
            token_end_index -= 1

        if not (offsets[token_start_index][0] <= start_char and offsets[token_end_index][1] >= end_char):
            start_positions.append(cls_index)
            end_positions.append(cls_index)
        else:
            while token_start_index < len(offsets) and offsets[token_start_index][0] <= start_char:
                token_start_index += 1
            start_positions.append(token_start_index - 1)

            while offsets[token_end_index][1] >= end_char:
                token_end_index -= 1
            end_positions.append(token_end_index + 1)

    tokenized_examples["start_positions"] = start_positions
    tokenized_examples["end_positions"] = end_positions
    return tokenized_examples

train_features = train_ds.map(
    prepare_train_features, batched=True, remove_columns=train_ds.column_names
)
val_features = val_ds.map(
    prepare_train_features, batched=True, remove_columns=val_ds.column_names
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_features,
    eval_dataset=val_features,
    data_collator=DefaultDataCollator(),
    processing_class=tokenizer,
)

trainer.train()

qa_pipe = pipeline(
    "question-answering",
    model=trainer.model,
    tokenizer=tokenizer,
    device=0 if DEVICE == "cuda" else -1,
)

# Retrieval corpus is built from unique training contexts only
train_contexts = list({ex["context"] for ex in train_examples})

def simple_tokenize(text):
    return normalize_text(text).split()

bm25_corpus = [simple_tokenize(c) for c in train_contexts]
bm25 = BM25Okapi(bm25_corpus)

dense_model = SentenceTransformer(DENSE_MODEL_NAME, device=DEVICE)
dense_embeddings = dense_model.encode(
    train_contexts,
    convert_to_numpy=True,
    show_progress_bar=True,
    normalize_embeddings=True,
)

def retrieve_bm25(question, top_k=TOP_K):
    scores = bm25.get_scores(simple_tokenize(question))
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [train_contexts[i] for i in top_idx], [float(scores[i]) for i in top_idx]

def retrieve_dense(question, top_k=TOP_K):
    q_emb = dense_model.encode([question], convert_to_numpy=True, normalize_embeddings=True)[0]
    sims = np.dot(dense_embeddings, q_emb)
    top_idx = np.argsort(sims)[::-1][:top_k]
    return [train_contexts[i] for i in top_idx], [float(sims[i]) for i in top_idx]

def retrieve_hybrid(question, top_k=TOP_K, alpha=0.5):
    bm25_scores = bm25.get_scores(simple_tokenize(question))
    q_emb = dense_model.encode([question], convert_to_numpy=True, normalize_embeddings=True)[0]
    dense_scores = np.dot(dense_embeddings, q_emb)

    def minmax(x):
        x = np.asarray(x, dtype=np.float32)
        mn, mx = x.min(), x.max()
        if mx - mn < 1e-9:
            return np.zeros_like(x)
        return (x - mn) / (mx - mn)

    hybrid_scores = alpha * minmax(bm25_scores) + (1 - alpha) * minmax(dense_scores)
    top_idx = np.argsort(hybrid_scores)[::-1][:top_k]
    return [train_contexts[i] for i in top_idx], [float(hybrid_scores[i]) for i in top_idx]

def has_answer(context, gold_answer):
    return normalize_text(gold_answer) in normalize_text(context)

def evaluate_gold_reader(test_examples):
    rows = []
    for ex in test_examples:
        pred = qa_pipe(question=ex["question"], context=ex["context"])["answer"]
        gold = ex["answers"]["text"][0]
        em, f1 = compute_em_f1(pred, gold)
        rows.append({"method": "FineTuned_GoldContext", "em": em, "f1": f1})
    return pd.DataFrame(rows)

def evaluate_rag_reader(test_examples, mode="bm25", top_k=TOP_K):
    rows = []
    qual = []

    for ex in test_examples:
        q = ex["question"]
        gold = ex["answers"]["text"][0]

        if mode == "bm25":
            retrieved, _ = retrieve_bm25(q, top_k=top_k)
        elif mode == "dense":
            retrieved, _ = retrieve_dense(q, top_k=top_k)
        elif mode == "hybrid":
            retrieved, _ = retrieve_hybrid(q, top_k=top_k)
        else:
            raise ValueError("Unknown mode")

        top1 = retrieved[0]
        pred = qa_pipe(question=q, context=top1)["answer"]
        em, f1 = compute_em_f1(pred, gold)

        r1 = float(any(has_answer(c, gold) for c in retrieved[:1]))
        r5 = float(any(has_answer(c, gold) for c in retrieved[:5]))

        rows.append({
            "method": mode,
            "em": em,
            "f1": f1,
            "recall@1": r1,
            "recall@5": r5,
        })

        qual.append({
            "method": mode,
            "question": q,
            "gold_answer": gold,
            "pred_answer": pred,
            "top1_contains_answer": has_answer(top1, gold),
            "top1_context": top1[:1000],
        })

    return pd.DataFrame(rows), pd.DataFrame(qual)

gold_df = evaluate_gold_reader(test_examples)
bm25_df, bm25_qual = evaluate_rag_reader(test_examples, "bm25")
dense_df, dense_qual = evaluate_rag_reader(test_examples, "dense")
hybrid_df, hybrid_qual = evaluate_rag_reader(test_examples, "hybrid")

summary_rows = [{
    "method": "FineTuned_GoldContext",
    "EM": gold_df["em"].mean(),
    "F1": gold_df["f1"].mean(),
    "Recall@1": np.nan,
    "Recall@5": np.nan,
}]

for name, df in [("BM25_RAG", bm25_df), ("Dense_RAG", dense_df), ("Hybrid_RAG", hybrid_df)]:
    summary_rows.append({
        "method": name,
        "EM": df["em"].mean(),
        "F1": df["f1"].mean(),
        "Recall@1": df["recall@1"].mean(),
        "Recall@5": df["recall@5"].mean(),
    })

summary = pd.DataFrame(summary_rows)
print("\nFinal Results")
print(summary)

summary.to_csv("final_results.csv", index=False)
pd.concat([bm25_qual, dense_qual, hybrid_qual], ignore_index=True).to_csv(
    "qualitative_examples.csv", index=False
)

print("\nSaved:")
print("- final_results.csv")
print("- qualitative_examples.csv")


DEVICE: cuda
Total QA examples: 23664
Train/Val/Test: 18931 2366 2367


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/72 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



Final Results
                  method        EM        F1  Recall@1  Recall@5
0  FineTuned_GoldContext  0.663709  0.835066       NaN       NaN
1               BM25_RAG  0.441910  0.565585  0.657372  0.813266
2              Dense_RAG  0.179975  0.251280  0.292776  0.479087
3             Hybrid_RAG  0.457964  0.585186  0.683143  0.843684

Saved:
- final_results.csv
- qualitative_examples.csv
